In [89]:
#[CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "ARDIENTE"
STUDENT_ID = "TUPM-26-1092"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])  # Use the last digit of the student ID as the seed
ID_SUM = sum(map(int, id_digits))
NAME_LEGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LEGTH  + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID_Sum:", ID_SUM)
print("Name Length:", NAME_LEGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

Student: ARDIENTE TUPM-26-1092
Seed Digit: 2
ID_Sum: 20
Name Length: 8
Random Seed: 261092
Verification Value: 162


In [90]:
#[CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Wokrspace:", WORK_DIR.resolve())
print("All dataset files verified.")

Wokrspace: C:\Users\ggard\Lab7_NumPy_Pandas_ARDIENTE
All dataset files verified.


In [91]:
#[CELL 3] NUMPY ARRAYS AND DATASET INSPECTION
demo = np.array([2,4,6,8])

print("Sample Array:", demo)
print("Element-wise Multiplication:", demo * 3)

raw_measurements = np.genfromtxt(measurement_path, dtype= float)
assert raw_measurements.ndim == 2

print("Shape:", raw_measurements.shape)
print("Dimesions:", raw_measurements.ndim)
print("Total Elements:", raw_measurements.size)
print("Data Type:", raw_measurements.dtype)
print("Missing Values:", np.isnan(raw_measurements).sum())


Sample Array: [2 4 6 8]
Element-wise Multiplication: [ 6 12 18 24]
Shape: (1567, 590)
Dimesions: 2
Total Elements: 924530
Data Type: float64
Missing Values: 41951


In [92]:
#[CELL 4] ARRAY SELECTION AND TRANSFORMATION
block = raw_measurements [:6,:4]
snapshot = block.copy()
block_copy = block.copy()

print("First Record:", block[0, :])
print("First Feature:", block[:,0])
print("Last Two Records:\n", block[-2, :])

reshaped = block_copy.reshape(3,8)

print("Selected Block Shape:", block.shape)
print("Copied Block Shape:", block_copy.shape)
print("Reshaped Shape:", reshaped.shape)
print("Transposed Shape:", block.T.shape)
print("Flattened Size:", block.flatten().size)

block_copy[0, 0] = 999.0
print("Original Unchanged:",
      np.array_equal(block, snapshot, equal_nan=True))



First Record: [3030.93   2564.     2187.7333 1411.1265]
First Feature: [3030.93 3095.78 2932.61 2988.72 3032.24 2946.25]
Last Two Records:
 [3032.24   2502.87   2233.3667 1326.52  ]
Selected Block Shape: (6, 4)
Copied Block Shape: (6, 4)
Reshaped Shape: (3, 8)
Transposed Shape: (4, 6)
Flattened Size: 24
Original Unchanged: True


In [93]:
#[CELL 5] MISSING VALUES AND NUMPY STATISTICS
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
    ).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

raw_usable = raw_measurements[:, usable_indices].copy()

feature_means = np.nanmean(raw_usable, axis=0)
feature_sds = np.nanstd(raw_usable, axis=0, ddof=1)

print("Total Missing Values:", missing.sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Retained Features:", len(usable_indices))
print("First Feature Mean:", f"{feature_means[0]:.4f}")
print("First Feature Sample SD:", f"{feature_sds[0]:.4f}")

Total Missing Values: 41951
Entirely Missing Features: 0
Constant Features: 116
Retained Features: 474
First Feature Mean: 3014.4529
First Feature Sample SD: 73.6218


In [94]:
#[CELL 6] IMPUTATION AND STANDARDIZATION
missing_before = np.isnan(raw_usable).sum()
feature_medians = np.nanmedian(raw_usable, axis=0)

prepared_np = np.where(
    np.isnan(raw_usable), feature_medians, raw_usable
 )

means = prepared_np.mean(axis=0)
sds = prepared_np.std(axis=0, ddof=1)
safe_sds = np.where(sds > 0, sds, 1.0)

standardized_np = (prepared_np - means) / safe_sds
flags = np.abs(standardized_np) > 3

print("Missing Before:", missing_before)
print("Missing After:", np.isnan(prepared_np).sum())
print("First Feature Median:", f"{feature_medians[0]:.4f}")
print("Standardize Mean:", f"{standardized_np[:,0].mean():.4f}")
print("Standardized SD:",  F" {standardized_np[:,0].std(ddof=1):.4f}")
print("Values Beyond ±3:", np.count_nonzero)


Missing Before: 41136
Missing After: 0
First Feature Median: 3011.4900
Standardize Mean: 0.0000
Standardized SD:  1.0000
Values Beyond ±3: <function count_nonzero at 0x000001147F6701A0>


In [95]:
#[CELL 7] FILTER LINES
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_filtered.txt"

with open(file_path, "r", encoding="utf-8") as source:
    for line in source:
        if "ID" in line or "Verification" in line:
            output.write(line)

print("Filtered File:", filtered_path)

NameError: name 'file_path' is not defined